# University of Kentucky Center for Poverty Research (UKCPR)

[National Welfare Data](https://ukcpr.uky.edu/resources)

In [1]:
import pandas as pd

# Set the maximum number of rows to display to 'None' (unlimited)
pd.set_option('display.max_rows', None)

# Optional: Also ensure column names don't get truncated
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [6]:
df = pd.read_excel("../../data/Nat_Welfare_Data_1980_2024.xlsx", sheet_name="Data")
df.head()

df.to_csv("../../data/ukcpr_raw.csv", index=False)

In [7]:
# filter for 1999-2016
df = df[df["year"].between(1999, 2016)].reset_index(drop=True)


In [8]:
# Reduce to key features
cols_to_keep = [
    'state_name', 'year',                          # Keys
    'Population',                                  # Calculate Rates
    'Unemployment rate', 'Poverty Rate',                # Core Economics
    'Food Stamp/SNAP Recipients', 'Gross State Product',        # Financial Stress
    'Medicaid beneficiaries',                      # Healthcare Access
    'Governor is Democrat (1=Yes)',                # Political Context
    'State Minimum Wage'                           # Economic Policy
]

df = df[cols_to_keep].copy()

col_names = ["state_abbr", "year", 
             "population",
             "unempl_rate", "poverty_rate",
             "snap_recipients", "gsp",
            "medicaid_beneficiaries",
            "gov_dem",
            "min_wage"]

df.columns = col_names

df.head()

,state_abbr,year,population,unempl_rate,poverty_rate,snap_recipients,gsp,medicaid_beneficiaries,gov_dem,min_wage
0,AL,1999,4369862,4.7,15.2,405273.0,115833.1,516011,1,5.15
1,AK,1999,619500,6.5,7.6,41262.0,24779.3,74095,1,5.65
2,AZ,1999,4778332,4.4,12.2,257362.0,155282.6,455723,0,5.15
3,AR,1999,2551373,4.6,14.7,252957.0,66791.2,379518,0,5.15
4,CA,1999,33145121,5.2,14.0,2027089.0,1247734.4,5063102,1,5.75


In [9]:
# Calculate rates and drop raw counts
df['snap_rate'] = df['snap_recipients'] / df['population']
df['medicaid_rate'] = df['medicaid_beneficiaries'] / df['population']

df = df.drop(columns=['snap_recipients', 'medicaid_beneficiaries', 'population'])
df.head()

,state_abbr,year,unempl_rate,poverty_rate,gsp,gov_dem,min_wage,snap_rate,medicaid_rate
0,AL,1999,4.7,15.2,115833.1,1,5.15,0.092743,0.118084
1,AK,1999,6.5,7.6,24779.3,1,5.65,0.066605,0.119605
2,AZ,1999,4.4,12.2,155282.6,0,5.15,0.053860,0.095373
3,AR,1999,4.6,14.7,66791.2,0,5.15,0.099145,0.14875
4,CA,1999,5.2,14.0,1247734.4,1,5.75,0.061158,0.152756


In [10]:
df.isna().sum()

state_abbr       0
year             0
unempl_rate      0
poverty_rate     0
gsp              0
gov_dem          0
min_wage         0
snap_rate        0
medicaid_rate    0
dtype: int64

In [11]:
# Add full state names
state_mapping = {
    'AL': 'Alabama', 'AK': 'Alaska', 'AZ': 'Arizona', 'AR': 'Arkansas', 
    'CA': 'California', 'CO': 'Colorado', 'CT': 'Connecticut', 'DE': 'Delaware', 
    'FL': 'Florida', 'GA': 'Georgia', 'HI': 'Hawaii', 'ID': 'Idaho', 
    'IL': 'Illinois', 'IN': 'Indiana', 'IA': 'Iowa', 'KS': 'Kansas', 
    'KY': 'Kentucky', 'LA': 'Louisiana', 'ME': 'Maine', 'MD': 'Maryland', 
    'MA': 'Massachusetts', 'MI': 'Michigan', 'MN': 'Minnesota', 'MS': 'Mississippi', 
    'MO': 'Missouri', 'MT': 'Montana', 'NE': 'Nebraska', 'NV': 'Nevada', 
    'NH': 'New Hampshire', 'NJ': 'New Jersey', 'NM': 'New Mexico', 'NY': 'New York', 
    'NC': 'North Carolina', 'ND': 'North Dakota', 'OH': 'Ohio', 'OK': 'Oklahoma', 
    'OR': 'Oregon', 'PA': 'Pennsylvania', 'RI': 'Rhode Island', 'SC': 'South Carolina', 
    'SD': 'South Dakota', 'TN': 'Tennessee', 'TX': 'Texas', 'UT': 'Utah', 
    'VT': 'Vermont', 'VA': 'Virginia', 'WA': 'Washington', 'WV': 'West Virginia', 
    'WI': 'Wisconsin', 'WY': 'Wyoming', 'DC': 'District of Columbia'
}

df['state_name'] = df['state_abbr'].map(state_mapping)

In [12]:
# Define the logical order
ordered_cols = [
    # 1. Identifiers
    'state_name', 'state_abbr', 'year', 
    
    # 2. Economic Features (Macro-level stress)
    'poverty_rate', 'unempl_rate', 'gsp', 'min_wage',
    
    # 3. Social Safety Net & Healthcare (Policy indicators)
    'snap_rate', 'medicaid_rate',
    
    # 4. Political Context
    'gov_dem'
]

# Apply the reordering
df = df[ordered_cols]

# Display the result
df.head()

,state_name,state_abbr,year,poverty_rate,unempl_rate,gsp,min_wage,snap_rate,medicaid_rate,gov_dem
0,Alabama,AL,1999,15.2,4.7,115833.1,5.15,0.092743,0.118084,1
1,Alaska,AK,1999,7.6,6.5,24779.3,5.65,0.066605,0.119605,1
2,Arizona,AZ,1999,12.2,4.4,155282.6,5.15,0.053860,0.095373,0
3,Arkansas,AR,1999,14.7,4.6,66791.2,5.15,0.099145,0.14875,0
4,California,CA,1999,14.0,5.2,1247734.4,5.75,0.061158,0.152756,1


In [13]:
# Save cleaned data
df.to_csv("../../data/UKCPR_cleaned.csv", index=False)